In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sn
import numpy as np
import json
from dotenv import load_dotenv
from sklearn import KNearestClassifier

# pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)  
# pd.set_option('display.max_colwidth', None)
data = pd.read_json('appraisals_dataset.json')


**Get all subjects for each appraisal**

In [ ]:
all_subjects = []
for appraisal in data['appraisals']:
    all_subjects.append(appraisal['subject'])
    
df_subjects = pd.DataFrame(all_subjects)
df_subjects.info()

**Get all properties**

In [ ]:
all_properties = []
for appraisal in data['appraisals']:
    for property in appraisal['properties']:
        all_properties.append(property)

df_properties = pd.DataFrame(all_properties)
df_properties.info()


**Select numerical property features** 

In [ ]:
df_properties.describe()
df_subjects.describe()

In [ ]:
df_properties[['gla', 'room_count', 'full_baths', 'half_baths', 'bedrooms', 'lot_size_sf', 'year_built']].info()

In [ ]:
df_properties.isnull().sum()

**Drop property columns with too many missing data**

In [27]:
df_properties.drop(['main_level_finished_area', 'bg_fin_area', 'upper_lvl_fin_area', 'public_remarks'], axis=1, inplace=True)

**Select relevant columns common to both subject and properties**

In [45]:
print(f"unique columns in subjects: {df_subjects.columns.__len__()}")
print(f"unique columns in properties: {df_properties.columns.__len__()}")


unique columns in subjects: 35
unique columns in properties: 24


In [54]:
props_selected = df_properties[['bedrooms', 'gla', 'year_built', 'structure_type', 'lot_size_sf', 'basement', 'heating', 'cooling', 'style']]
subs_selected = df_subjects[['num_beds', 'gla', 'year_built', 'structure_type', 'lot_size_sf', 'basement', 'heating','cooling', 'style']]

In [97]:
#set missing bedroom values with median
props_selected['bedrooms']
median = props_selected['bedrooms'].median()
props_selected.fillna({'bedrooms': median}, inplace=True)


C:\Users\dejhs\AppData\Local\Temp\ipykernel_36168\258159172.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  props_selected.fillna({'bedrooms': median}, inplace=True)


In [77]:
#convert number of beds for selected subjects to floats
subs_selected['num_beds'] = pd.to_numeric(subs_selected['num_beds'], errors='coerce')

C:\Users\dejhs\AppData\Local\Temp\ipykernel_36168\2518629405.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  subs_selected['num_beds'] = pd.to_numeric(subs_selected['num_beds'], errors='coerce')


In [ ]:
#find out how many num_beds were not able to be converted
subs_selected['num_beds'].info()

#fill null values of num_beds with median
subs_selected['num_beds'] = subs_selected['num_beds'].fillna(subs_selected['num_beds'].median())


0     3.0
1     3.0
2     3.0
3     3.0
4     4.0
5     3.0
6     3.0
7     5.0
8     4.0
9     4.0
10    4.0
11    4.0
12    4.0
13    3.0
14    2.0
15    3.0
16    2.0
17    2.0
18    3.0
19    3.0
Name: bedrooms, dtype: float64